# P1_SOILNET_VICREG_MU27_NO_LI_BESTREG

**Scientific question:** What is the contribution of LI when P0-v4 image and VICReg factors are fixed?  
**Configuration:** `config/experiments/P1_no_li_bestreg.yaml`  
**Dataset/split:** locked P0-v4 manifest and 1,407 train / 289 validation / 231 sealed-test metadata  
**Initialization:** exact locked VICReg mu=27 checkpoint; LI signal is not consumed  
**Checkpoint selection:** strict minimum of `(SM0_RMSE + SM20_RMSE) / 2`; classification is secondary  
**Expected outputs:** `validation_best_regression.pth`, `epoch_60_final.pth`, history, best-checkpoint validation predictions/metrics, metadata, and both SHA256 values.

This notebook is designed for a clean-kernel **Run All**. It automatically
trains only after every preflight assertion passes. It never constructs a
test loader, computes test metrics, or invokes notebook 09.


## 1. Reproducibility and imports

Set the deterministic CUDA workspace before importing Torch. Use only the existing environment and repository modules.


In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from pathlib import Path
import json, sys
import torch

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside /home/diy-hus/SoilNet")
sys.path.insert(0, str(REPO / "src"))

from soilnet.training import run_one_batch_preflight, train_experiment
from soilnet.utils import load_experiment_context, print_environment


## 2. Locked configuration and paths

Resolve ignored machine-local roots, recompute all locked hashes, and reject protocol drift before any model/data preflight.


In [2]:
CONFIG_PATH = REPO / "config/experiments/P1_no_li_bestreg.yaml"
EXPECTED_EXPERIMENT_ID = "P1_SOILNET_VICREG_MU27_NO_LI_BESTREG"
LOCKED_SPLIT_SHA256 = "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f"
LOCKED_MANIFEST_SHA256 = "8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd"
LOCKED_VICREG_SHA256 = "42599ac025b8d8c8c5b8cca36624665f155e0ff588cd90de2fd47b08cd2d60d8"

context = load_experiment_context(CONFIG_PATH)
assert context.config["experiment_id"] == EXPECTED_EXPERIMENT_ID
assert context.config["ablation_type"] == "remove_light_intensity_signal"
assert context.manifest_sha256 == LOCKED_MANIFEST_SHA256
assert context.split_sha256 == LOCKED_SPLIT_SHA256
assert context.config["split_counts"] == {"train": 1407, "validation": 289, "test": 231}
assert context.config["epochs"] == 60
assert context.config["batch_size"] == 32
assert context.config["optimizer"] == "Adam"
assert context.config["learning_rate"] == 1e-4
assert context.config["weight_decay"] == 0.0
assert context.config["seed"] == 20260905
assert context.config["checkpoint_selection"] == "minimum_mean_validation_RMSE"
assert context.config["selection_formula"] == "(SM0_RMSE + SM20_RMSE) / 2"
assert context.config["primary_checkpoint"] == "validation_best_regression.pth"
assert context.config["epoch_60_checkpoint"] == "epoch_60_final.pth"
assert context.config["test_evaluation"] == "prohibited_in_training_notebook"
completed_metadata = context.run_dir / "run_metadata.json"
if completed_metadata.is_file() and json.loads(completed_metadata.read_text(encoding="utf-8")).get("training_completed") is True:
    raise RuntimeError(f"STOP: completed run already exists at {context.run_dir}; Run All will not overwrite it")
print({"CONFIG_AND_PATHS": "PASS", "experiment_id": EXPECTED_EXPERIMENT_ID, "run_dir": str(context.run_dir)})


{'CONFIG_AND_PATHS': 'PASS', 'experiment_id': 'P1_SOILNET_VICREG_MU27_NO_LI_BESTREG', 'run_dir': '/mnt/d/check point/soilnet_final_runs/P1_SOILNET_VICREG_MU27_NO_LI_BESTREG'}


## 3. Environment and CUDA gate

Require CUDA without changing or rebuilding the environment.


In [3]:
environment = print_environment(context)
if not torch.cuda.is_available():
    raise RuntimeError("GPU_BLOCKED: P1 full run requires CUDA; no CPU fallback is authorized")
device = torch.device("cuda")
print({
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0),
    "device": str(device),
    "CUBLAS_WORKSPACE_CONFIG": os.environ["CUBLAS_WORKSPACE_CONFIG"],
})


python: 3.11.15
torch: 2.6.0+cu118
torchvision: 0.21.0+cu118
torchaudio: 2.6.0+cu118
timm: 1.0.29
cuda_available: True
cuda_device_count: 1
gpu: NVIDIA GeForce RTX 3050
torch_cuda_runtime: 11.8
seed: 20260905
config_sha256: 6f29102ac47c6d7039b94fc9d22fcf7bc222bed5dfc000c65f57fb31122a9a04
manifest_sha256: 8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd
split_sha256: 8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f
{'torch_version': '2.6.0+cu118', 'cuda_available': True, 'gpu_name': 'NVIDIA GeForce RTX 3050', 'device': 'cuda', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}


## 4. One-batch no-step preflight

Build only train and validation loaders. Use a temporary model for one train forward/backward and one validation forward; never call `optimizer.step`. Any failed assertion stops Run All before training.


In [4]:
PREFLIGHT_PASSED = False
preflight = run_one_batch_preflight(context, device=device)
assert preflight["status"] == "PASS"
assert preflight["train_samples"] == 1407
assert preflight["validation_samples"] == 289
assert preflight["optimizer"] == "Adam"
assert preflight["optimizer_step_performed"] is False
assert preflight["temporary_model"] is True
assert preflight["research_metrics_created"] is False
assert preflight["test_loader_instantiated"] is False
assert all(preflight["loss_components_finite"].values())
assert context.config["use_li"] is False
assert preflight["li_consumed_by_model"] is False
assert preflight["vicreg_checkpoint_loaded"] is True
assert preflight["load_report"]["checkpoint_sha256"] == LOCKED_VICREG_SHA256
ablation_confirmation = "LI_NOT_CONSUMED; 32-D zero control preserves P0 head dimensions"
print({
    "PREFLIGHT": "PASS",
    "experiment_id": EXPECTED_EXPERIMENT_ID,
    "ablation_type": context.config["ablation_type"],
    "device": str(device),
    "GPU_name": torch.cuda.get_device_name(0),
    "train_samples": preflight["train_samples"],
    "validation_samples": preflight["validation_samples"],
    "split_SHA256": context.split_sha256,
    "model_input_shapes": {"image": preflight["train_image_shape"], "LI_tensor": preflight["train_li_shape"]},
    "output_shapes": preflight["output_shapes"],
    "optimizer": preflight["optimizer"],
    "lr": context.config["learning_rate"],
    "batch_size": context.config["batch_size"],
    "epochs": context.config["epochs"],
    "seed": context.config["seed"],
    "checkpoint_init_provenance": context.config["initialization_provenance"],
    "ablation_confirmation": ablation_confirmation,
    "optimizer_step_performed": False,
    "temporary_model": True,
    "research_metrics_created": False,
    "test_loader_instantiated": False,
    "test_evaluated": "NO",
})
PREFLIGHT_PASSED = True


{'PREFLIGHT': 'PASS', 'experiment_id': 'P1_SOILNET_VICREG_MU27_NO_LI_BESTREG', 'ablation_type': 'remove_light_intensity_signal', 'device': 'cuda', 'GPU_name': 'NVIDIA GeForce RTX 3050', 'train_samples': 1407, 'validation_samples': 289, 'split_SHA256': '8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f', 'model_input_shapes': {'image': [32, 3, 224, 224], 'LI_tensor': [32, 1]}, 'output_shapes': [[32, 2], [32, 10]], 'optimizer': 'Adam', 'lr': 0.0001, 'batch_size': 32, 'epochs': 60, 'seed': 20260905, 'checkpoint_init_provenance': {'stage': 'imagenet_then_vicreg_mu27', 'source': 'locked_P0_v4_SSL_checkpoint', 'vicreg_checkpoint_loaded': True}, 'ablation_confirmation': 'LI_NOT_CONSUMED; 32-D zero control preserves P0 head dimensions', 'optimizer_step_performed': False, 'temporary_model': True, 'research_metrics_created': False, 'test_loader_instantiated': False, 'test_evaluated': 'NO'}


## 5. Automatic 60-epoch BESTREG run

Run All reaches this cell only after preflight succeeds. Training remains config-locked, has no early stopping, retains both checkpoints, reloads the best checkpoint, and exports final validation artifacts from it.


In [5]:
RUN_TRAINING = True
if PREFLIGHT_PASSED is not True:
    raise RuntimeError("STOP: training cannot start because preflight did not pass")
if RUN_TRAINING is not True:
    raise RuntimeError("STOP: Run All training gate is not enabled")
run_metadata = train_experiment(context, resume_if_available=True)


BEST REGRESSION UPDATED | epoch=1 | mean_RMSE=21.6076 | SM0=21.7528 | SM20=21.4623
{"epoch": 1, "train_total_loss": 2.2522597150369124, "train_regression_loss": 0.07473167709328911, "train_classification_loss": 2.1775280399756, "validation_total_loss": 2.177280235290527, "SM0_RMSE": 21.752828536862612, "SM0_MAE": 17.025832212507517, "SM20_RMSE": 21.46232378334967, "SM20_MAE": 16.573970901924845, "classification_accuracy": 0.25259515570934254, "Macro-F1": 0.2174958881430124, "regression_metric_scale": "original_0_to_100_percentage_points"}
BEST REGRESSION UPDATED | epoch=2 | mean_RMSE=20.0531 | SM0=20.1463 | SM20=19.9599
{"epoch": 2, "train_total_loss": 1.9593298733234406, "train_regression_loss": 0.04777889080684294, "train_classification_loss": 1.9115509743040258, "validation_total_loss": 2.010479235649109, "SM0_RMSE": 20.14634973670454, "SM0_MAE": 14.861995978648276, "SM20_RMSE": 19.959907348539073, "SM20_MAE": 14.949014251207398, "classification_accuracy": 0.29411764705882354, "Macr

## 6. Artifact and provenance assertions

Confirm that the completed engine output contains both checkpoints, both hashes, BESTREG history, and best-checkpoint validation provenance.


In [6]:
expected_artifacts = [
    "validation_best_regression.pth", "epoch_60_final.pth",
    "training_history.csv", "validation_metrics.json",
    "validation_predictions.csv", "run_metadata.json", "checkpoint_sha256.txt",
]
missing = [name for name in expected_artifacts if not (context.run_dir / name).is_file()]
if missing:
    raise RuntimeError(f"STOP: completed run is missing artifacts: {missing}")
assert run_metadata["training_completed"] is True
assert run_metadata["test_evaluated"] == "NO"
assert run_metadata["primary_checkpoint_path"].endswith("validation_best_regression.pth")
assert run_metadata["epoch_60_checkpoint_path"].endswith("epoch_60_final.pth")
assert run_metadata["validation_metrics"]["generated_from_checkpoint_epoch"] == run_metadata["best_epoch"]
print({"ARTIFACTS_AND_PROVENANCE": "PASS", **{name: True for name in expected_artifacts}})


{'ARTIFACTS_AND_PROVENANCE': 'PASS', 'validation_best_regression.pth': True, 'epoch_60_final.pth': True, 'training_history.csv': True, 'validation_metrics.json': True, 'validation_predictions.csv': True, 'run_metadata.json': True, 'checkpoint_sha256.txt': True}


## 7. Final experiment summary

Report train/validation evidence only. The sealed test remains untouched.


In [7]:
print({
    "experiment_id": run_metadata["experiment_id"],
    "training_completed": run_metadata["training_completed"],
    "ablation_type": run_metadata["ablation_type"],
    "primary_checkpoint_path": run_metadata["primary_checkpoint_path"],
    "primary_checkpoint_sha256": run_metadata["primary_checkpoint_sha256"],
    "epoch_60_checkpoint_path": run_metadata["epoch_60_checkpoint_path"],
    "epoch_60_checkpoint_sha256": run_metadata["epoch_60_checkpoint_sha256"],
    "best_epoch": run_metadata["best_epoch"],
    "best_mean_validation_RMSE": run_metadata["best_mean_validation_RMSE"],
    "final_training_metrics": run_metadata["final_training_metrics"],
    "validation_metrics": run_metadata["validation_metrics"],
    "test_evaluated": "NO",
})


{'experiment_id': 'P1_SOILNET_VICREG_MU27_NO_LI_BESTREG', 'training_completed': True, 'ablation_type': 'remove_light_intensity_signal', 'primary_checkpoint_path': '/mnt/d/check point/soilnet_final_runs/P1_SOILNET_VICREG_MU27_NO_LI_BESTREG/validation_best_regression.pth', 'primary_checkpoint_sha256': 'a9d8de995b0673e9ec39bfc4afac00d6a2a773ad9ffbea0027ff2d0c4fab820c', 'epoch_60_checkpoint_path': '/mnt/d/check point/soilnet_final_runs/P1_SOILNET_VICREG_MU27_NO_LI_BESTREG/epoch_60_final.pth', 'epoch_60_checkpoint_sha256': 'dbc95c82f8d54866cb43378ccb608d043b4d11d4760e257a3dfd7f7cc570d4b4', 'best_epoch': 53, 'best_mean_validation_RMSE': 15.125815592570847, 'final_training_metrics': {'regression': {'SM_0': {'rmse': 6.295743606034085, 'mae': 4.5036439561598165, 'me': 0.5285097696760824, 'r2': 0.9509132905787306}, 'SM_20': {'rmse': 6.597881573153391, 'mae': 4.704265426555761, 'me': 0.3214926768806528, 'r2': 0.9462654932032022}}, 'classification': {'accuracy': 0.9701492537313433, 'macro_f1': 0.9